# 02 · Silver layer: validated, keyed star schema

**Project:** IPL data engineering (Cricsheet ball-by-ball data, 2008–2026)
**Layer:** silver. The 11 bronze tables are validated and copied into `workspace.silver.<table>` with **explicit column lists**, **NOT NULL key columns** and **primary keys**. The star schema itself (dimensions, facts, bridge, aliases) was designed in the Python pipeline, so silver does not reshape the data; its job is to **prove the data is correct and declare its keys** before gold queries depend on it.

**Notebook flow**
1. **Part A · Validate bronze.** Value-level checks, foreign-key (orphan) checks and key-readiness checks, all against `workspace.bronze`. Bronze only proved that nothing was dropped by type casts; this proves the values make sense.
2. **Part B · Build silver.** Per table: build → key check → `NOT NULL` on key columns → `PRIMARY KEY` → `DESCRIBE`.
3. **Part C · Final verification.** Every table has a key, and row counts match bronze.
4. **Appendix.** A demonstration of what Unity Catalog constraints do and do not enforce.

**Conventions**
- **Columns are listed by name**, never `SELECT *`. Names and order come from `information_schema.columns`, so the silver schema is explicit and cannot change silently if bronze changes.
- **Ids from outside** (`match_id`, player ids) are `STRING`. **Ids we generate** (season, team, venue, `date_key`) are `INT`.
- **Unity Catalog enforces only `NOT NULL`.** `PRIMARY KEY` is informational: a duplicate insert still succeeds. Uniqueness is therefore checked by comparing `count(*)`, `count(key)` and `count(DISTINCT key)`.
- **`CREATE OR REPLACE TABLE ... AS SELECT` drops all constraints**, so constraints are always added *after* the load.
- Foreign keys are verified with orphan queries (Part A) and are not declared as constraints.
- Run the notebook top to bottom. Every build cell is `CREATE OR REPLACE`, so it is safe to rerun. An `ADD CONSTRAINT` cell run **alone** a second time fails with `CONSTRAINT_ALREADY_EXISTS`; rerun the build cell above it first.
- The **last cell** of the notebook is an expected failure (see the appendix).

| Table | Rows |
|---|---|
| dim_season | 19 |
| dim_team | 15 |
| dim_venue | 36 |
| dim_date | 6,618 |
| dim_player | 964 |
| fact_match | 1,243 |
| fact_delivery | 295,732 |
| bridge_wicket_fielder | 11,266 |
| season_alias | 19 |
| team_alias | 19 |
| venue_alias | 60 |

# Part A · Validate bronze

## A1 · Value-level checks on `fact_delivery`
Run totals and wicket totals are the numbers every gold query will be built on, so they are reconciled first.

**Check 1: the two ways of totalling runs agree.** `runs_total` should equal `runs_batter + runs_extras`, both in aggregate and row by row.

In [0]:
SELECT
  sum(runs_total) AS runs_total,
  sum(runs_batter) + sum(runs_extras) AS total_runs
FROM
  workspace.bronze.fact_delivery;

runs_total,total_runs
401738,401738


Row-level version: count the deliveries where `runs_total <> runs_batter + runs_extras`. Expected **0**.

In [0]:
SELECT
  count(*) AS total_rows
FROM
  workspace.bronze.fact_delivery
WHERE
  runs_total <> runs_batter + runs_extras;

total_rows
0


**Check 2: wicket and legal-delivery totals.** These are the reference numbers from the Python pipeline.
- `wickets` counts every dismissal. `bowler_wickets` counts only the ones credited to the bowler, so the **1,223** difference are dismissals such as run-outs, which must not count in a bowler's tally.
- `illegal_deliveries` (wides and no-balls) matter later: balls faced and overs bowled must exclude them.
- Use `count_if(...)` here because `SUM` does not work on `BOOLEAN`, and `COUNT(col)` would count every non-NULL value, `false` included.

In [0]:
SELECT
  count_if(is_wicket) AS wickets,
  count_if(is_bowler_wicket) AS bowler_wickets,
  count_if(NOT is_legal_delivery) AS illegal_deliveries
FROM
  workspace.bronze.fact_delivery;

wickets,bowler_wickets,illegal_deliveries
14705,13482,11102


## A2 · Foreign-key (orphan) checks

An **orphan** is a child row whose parent does not exist, for example a delivery pointing at a player who is not in `dim_player`. Orphans make joins silently drop rows, so every reference is checked. The pattern is the same everywhere: `LEFT JOIN` child → parent, then keep the rows where **the parent key IS NULL**. Expected **0** each time.

**Nullable references:** if the child column can legitimately be NULL, a NULL has no parent and would be counted as an orphan. For those columns the query adds `child_col IS NOT NULL`.

### `fact_delivery` → its parents
A delivery references a match, four players and the batting team. First confirm the match key is not missing, then check each reference.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_delivery AS fd
    LEFT JOIN workspace.bronze.fact_match AS fm
      ON fd.match_id = fm.match_id
WHERE
  fm.match_id IS NULL;

orphan_rows
0


`batter_id` → `dim_player`.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_delivery AS f
    LEFT JOIN workspace.bronze.dim_player AS p
      ON f.batter_id = p.player_id
WHERE
  p.player_id IS NULL;

orphan_rows
0


`bowler_id` → `dim_player`.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_delivery AS f
    LEFT JOIN workspace.bronze.dim_player AS p
      ON f.bowler_id = p.player_id
WHERE
  p.player_id IS NULL;

orphan_rows
0


`non_striker_id` → `dim_player`.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_delivery AS f
    LEFT JOIN workspace.bronze.dim_player as p
      ON f.non_striker_id = p.player_id
WHERE
  p.player_id IS NULL;

orphan_rows
0


`player_out_id` → `dim_player`. This column is NULL on non-wicket deliveries (14,705 values are filled), so the query adds `player_out_id IS NOT NULL`.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_delivery as f
    LEFT JOIN workspace.bronze.dim_player AS p
      ON f.player_out_id = p.player_id
WHERE
  f.player_out_id IS NOT NULL
  AND p.player_id IS NULL;

orphan_rows
0


`batting_team_id` → `dim_team`. First confirm the column has no NULLs, then run the orphan check.

In [0]:
SELECT
  count(*) AS total_rows,
  count_if(batting_team_id IS NULL) AS batting_team_nulls
FROM
  workspace.bronze.fact_delivery;

total_rows,batting_team_nulls
295732,0


In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_delivery f
    LEFT JOIN workspace.bronze.dim_team t
      ON f.batting_team_id = t.team_id
WHERE
  t.team_id IS NULL;

orphan_rows
0


### `bridge_wicket_fielder` → its parents
The bridge points at one delivery (a composite key of three columns, so all three appear in the join) and one fielder.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.bridge_wicket_fielder as b
    LEFT JOIN workspace.bronze.fact_delivery AS f
      ON b.match_id = f.match_id
      AND b.innings_number = f.innings_number
      AND b.delivery_seq = f.delivery_seq
WHERE
  f.match_id IS NULL;

orphan_rows
0


In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.bridge_wicket_fielder AS b
    LEFT JOIN workspace.bronze.dim_player AS p
      ON b.fielder_id = p.player_id
WHERE
  p.player_id IS NULL;

orphan_rows
0


### `fact_match` → the dimensions
First count the NULLs in every foreign-key column. This decides which orphan checks need `IS NOT NULL`:
- `winner_team_id` has **25** NULLs: ties and no-result matches have no winner.
- `eliminator_team_id` has **1,227** NULLs: it is filled only for the 16 matches decided by a super over.
- All other references are fully populated (0 NULLs), so they need no extra condition.

In [0]:
SELECT
  count(*) AS total_rows,
  count_if(season_id IS NULL) AS season_nulls,
  count_if(venue_id IS NULL) AS venue_nulls,
  count_if(team1_id IS NULL) AS team1_nulls,
  count_if(team2_id IS NULL) AS team2_nulls,
  count_if(winner_team_id IS NULL) AS winner_nulls,
  count_if(eliminator_team_id IS NULL) AS eliminator_nulls,
  count_if(date_key IS NULL) AS date_nulls
FROM
  workspace.bronze.fact_match;

total_rows,season_nulls,venue_nulls,team1_nulls,team2_nulls,winner_nulls,eliminator_nulls,date_nulls
1243,0,0,0,0,25,1227,0


`season_id` → `dim_season`.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_match m
    LEFT JOIN workspace.bronze.dim_season s
      ON m.season_id = s.season_id
WHERE
  s.season_id IS NULL;

orphan_rows
0


`venue_id` → `dim_venue`.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_match m
    LEFT JOIN workspace.bronze.dim_venue v
      ON m.venue_id = v.venue_id
WHERE
  v.venue_id IS NULL;

orphan_rows
0


`team1_id` → `dim_team`.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_match m
    LEFT JOIN workspace.bronze.dim_team t
      ON m.team1_id = t.team_id
WHERE
  t.team_id IS NULL;

orphan_rows
0


`team2_id` → `dim_team`. Both teams of a match reference the same dimension, so both columns need the check.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_match m
    LEFT JOIN workspace.bronze.dim_team t
      ON m.team2_id = t.team_id
WHERE
  t.team_id IS NULL;

orphan_rows
0


`winner_team_id` → `dim_team` (nullable, so `IS NOT NULL` is added).

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_match m
    LEFT JOIN workspace.bronze.dim_team t
      ON m.winner_team_id = t.team_id
WHERE
  m.winner_team_id IS NOT NULL
  AND t.team_id IS NULL;

orphan_rows
0


`eliminator_team_id` → `dim_team` (nullable, so `IS NOT NULL` is added).

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_match m
    LEFT JOIN workspace.bronze.dim_team t
      ON m.eliminator_team_id = t.team_id
WHERE
  m.eliminator_team_id IS NOT NULL
  AND t.team_id IS NULL;

orphan_rows
0


`date_key` → `dim_date`.

In [0]:
SELECT
  count(*) AS orphan_rows
FROM
  workspace.bronze.fact_match m
    LEFT JOIN workspace.bronze.dim_date d
      ON m.date_key = d.date_key
WHERE
  d.date_key IS NULL;

orphan_rows
0


## A3 · Key readiness

Before a column is declared a primary key it must be **never NULL** and **unique**. With the pattern `count(*)` = `count(key)` = `count(DISTINCT key)`, the three numbers must be equal; any gap shows NULLs or duplicates.

### Dimensions
`dim_season`, `dim_team`, `dim_venue`, `dim_date` and `dim_player`: 19, 15, 36, 6,618 and 964 rows, all keys non-null and distinct.

In [0]:
SELECT
  count(*) AS total_rows,
  count(season_id) AS non_null_keys,
  count(DISTINCT season_id) AS distinct_keys
FROM
  workspace.bronze.dim_season;

total_rows,non_null_keys,distinct_keys
19,19,19


In [0]:
SELECT
  count(*) AS total_rows,
  count(team_id) AS non_null_keys,
  count(DISTINCT team_id) AS distinct_keys
FROM
  workspace.bronze.dim_team;

total_rows,non_null_keys,distinct_keys
15,15,15


In [0]:
SELECT
  count(*) AS total_rows,
  count(venue_id) AS non_null_keys,
  count(DISTINCT venue_id) AS distinct_keys
FROM
  workspace.bronze.dim_venue;

total_rows,non_null_keys,distinct_keys
36,36,36


In [0]:
SELECT
  count(*) AS total_rows,
  count(date_key) AS non_null_keys,
  count(DISTINCT date_key) AS distinct_keys
FROM
  workspace.bronze.dim_date;

total_rows,non_null_keys,distinct_keys
6618,6618,6618


In [0]:
SELECT
  count(*) AS total_rows,
  count(player_id) as non_null_ids,
  count(DISTINCT player_id) AS distinct_ids
FROM
  workspace.bronze.dim_player;

total_rows,non_null_ids,distinct_ids
964,964,964


`dim_date` has one more rule: `date_key` is the date written as `yyyyMMdd`, so it must agree with `full_date`. **0 mismatches** and 6,618 distinct dates also confirm there is exactly one row per calendar day.

In [0]:
SELECT
  count(*) AS total_rows,
  count(DISTINCT full_date) AS distinct_dates,
  count_if(date_key <> CAST(date_format(full_date, 'yyyyMMdd') AS INT)) AS mismatched_keys
FROM
  workspace.bronze.dim_date;

total_rows,distinct_dates,mismatched_keys
6618,6618,0


### `fact_match`
One row per match: 1,243 rows, 1,243 non-null `match_id` values, 1,243 distinct.

In [0]:
SELECT
  count(*) AS total_rows,
  count(match_id) AS match_id_count,
  count(DISTINCT match_id) AS match_id_distinct
FROM
  workspace.bronze.fact_match;

total_rows,match_id_count,match_id_distinct
1243,1243,1243


### `fact_delivery`
The grain is one row per delivery, so the key is **composite**: `match_id + innings_number + delivery_seq`. No part of it may be NULL (all four counts are 295,732), and no combination may repeat. The `GROUP BY ... HAVING count(*) > 1` query lists duplicated keys, so an **empty result means no duplicates**.

In [0]:
SELECT
  count(*) AS total_rows,
  count(match_id) AS match_id,
  count(innings_number) AS innings_number,
  count(delivery_seq) AS delivery_seq
FROM
  workspace.bronze.fact_delivery

total_rows,match_id,innings_number,delivery_seq
295732,295732,295732,295732


In [0]:
SELECT
  count(*) AS total_rows,
  match_id,
  innings_number,
  delivery_seq
FROM
  workspace.bronze.fact_delivery
GROUP BY
  match_id,
  innings_number,
  delivery_seq
HAVING
  count(*) > 1;

total_rows,match_id,innings_number,delivery_seq


# Part B · Build silver

## 0 · Setup: schema
Silver tables live in their own schema, `workspace.silver`.

In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.silver;

## 1 · Dimension tables

**Recipe for every table:** build with an explicit column list → key check (`count(*)`, `count(key)`, `count(DISTINCT key)`) → `SET NOT NULL` on the key column → `ADD CONSTRAINT ... PRIMARY KEY` → `DESCRIBE`.

Primary-key columns must be `NOT NULL` first, and constraints go on *after* the load, because `CREATE OR REPLACE TABLE` removes them.

The column names and types for the four small dimensions come from `information_schema.columns`:

In [0]:
SELECT
  table_name,
  ordinal_position,
  column_name,
  data_type
FROM
  workspace.information_schema.columns
WHERE
  table_schema = 'bronze'
  AND table_name IN ('dim_season', 'dim_team', 'dim_venue', 'dim_date')
ORDER BY
  table_name,
  ordinal_position;

table_name,ordinal_position,column_name,data_type
dim_date,0,date_key,INT
dim_date,1,full_date,DATE
dim_date,2,year,INT
dim_date,3,month,INT
dim_date,4,day,INT
dim_date,5,weekday_name,STRING
dim_date,6,is_weekend,BOOLEAN
dim_season,0,season_id,INT
dim_season,1,season_year,INT
dim_team,0,team_id,INT


### 1.1 · dim_season
Primary key `season_id`. The key check expects 19 rows and 19 distinct keys. Constraints are added in the cell after it; `DESCRIBE TABLE EXTENDED` then lists `pk_dim_season`.

In [0]:
CREATE OR REPLACE TABLE workspace.silver.dim_season AS
SELECT
  season_id,
  season_year
FROM
  workspace.bronze.dim_season;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  count(*) AS total_rows,
  count(DISTINCT season_id) AS distinct_keys
FROM
  workspace.silver.dim_season;

total_rows,distinct_keys
19,19


In [0]:
ALTER TABLE
  workspace.silver.dim_season
ALTER COLUMN
  season_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.dim_season
ADD
  CONSTRAINT pk_dim_season PRIMARY KEY (season_id);

`DESCRIBE TABLE EXTENDED` shows the schema and the `pk_dim_season` constraint. (The Appendix uses this table to show what the constraints do and do not enforce.)

In [0]:
DESCRIBE TABLE EXTENDED workspace.silver.dim_season;

col_name,data_type,comment
season_id,int,null
season_year,int,null
# Clustering Information,,
# col_name,data_type,comment
,,
# Delta Statistics Columns,,
Column Names,"season_id, season_year",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


### 1.2 · dim_team
Primary key `team_id`: 15 rows, all keys non-null and distinct.

In [0]:
CREATE OR REPLACE TABLE workspace.silver.dim_team AS
SELECT
  team_id,
  team_name
FROM
  workspace.bronze.dim_team;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  count(*) AS total_rows,
  count(team_id) AS non_null_keys,
  count(DISTINCT team_id) AS distinct_keys
FROM
  workspace.silver.dim_team;

total_rows,non_null_keys,distinct_keys
15,15,15


In [0]:
ALTER TABLE
  workspace.silver.dim_team
ALTER COLUMN
  team_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.dim_team
ADD
  CONSTRAINT pk_dim_team PRIMARY KEY (team_id);

In [0]:
DESCRIBE TABLE EXTENDED workspace.silver.dim_team;

col_name,data_type,comment
team_id,int,null
team_name,string,null
# Clustering Information,,
# col_name,data_type,comment
,,
# Delta Statistics Columns,,
Column Names,"team_id, team_name",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


### 1.3 · dim_venue
Primary key `venue_id`: 36 rows. Besides the key, the venue **names** are checked: non-null, unique and not blank (`trim(venue_name) = ''`), because gold will show the name as the label.

In [0]:
CREATE OR REPLACE TABLE workspace.silver.dim_venue AS
SELECT
  venue_id,
  venue_name
FROM
  workspace.bronze.dim_venue;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  count(*) AS total_rows,
  count(venue_id) AS non_null_keys,
  count(DISTINCT venue_id) AS distinct_keys
FROM
  workspace.silver.dim_venue;

total_rows,non_null_keys,distinct_keys
36,36,36


In [0]:
SELECT
  count(*) AS total_rows,
  count(venue_name) AS non_null_names,
  count(DISTINCT venue_name) AS distinct_names,
  count_if(trim(venue_name) = '') AS blank_names
FROM
  workspace.silver.dim_venue;

total_rows,non_null_names,distinct_names,blank_names
36,36,36,0


Add `NOT NULL` and the primary key, then `DESCRIBE`.

In [0]:
ALTER TABLE
  workspace.silver.dim_venue
ALTER COLUMN
  venue_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.dim_venue
ADD
  CONSTRAINT pk_dim_venue PRIMARY KEY (venue_id);

DESCRIBE TABLE EXTENDED workspace.silver.dim_venue;

col_name,data_type,comment
venue_id,int,null
venue_name,string,null
# Clustering Information,,
# col_name,data_type,comment
,,
# Delta Statistics Columns,,
Column Names,"venue_id, venue_name",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


### 1.4 · dim_date
Primary key `date_key`: 6,618 rows. This table has a `DATE` column and a `BOOLEAN` flag, so the stored types are worth confirming in the `DESCRIBE`.

In [0]:
CREATE OR REPLACE TABLE workspace.silver.dim_date AS
SELECT
  date_key,
  full_date,
  year,
  month,
  day,
  weekday_name,
  is_weekend
FROM
  workspace.bronze.dim_date;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  count(*) AS total_rows,
  count(date_key) AS non_null_keys,
  count(DISTINCT date_key) AS distinct_keys
FROM
  workspace.silver.dim_date;

total_rows,non_null_keys,distinct_keys
6618,6618,6618


The derived columns must agree with `full_date`: year, month, day, weekday name and the weekend flag. `dayofweek()` returns 1 for Sunday and 7 for Saturday, so the weekend test is `IN (1, 7)`. Expected **0** mismatches in every column.

In [0]:
SELECT
  count_if(year <> year(full_date)) AS bad_year,
  count_if(month <> month(full_date)) AS bad_month,
  count_if(day <> day(full_date)) AS bad_day,
  count_if(weekday_name <> date_format(full_date, 'EEEE')) AS bad_weekday,
  count_if(is_weekend <> (dayofweek(full_date) in (1, 7))) AS bad_weekend
FROM
  workspace.silver.dim_date;

bad_year,bad_month,bad_day,bad_weekday,bad_weekend
0,0,0,0,0


In [0]:
ALTER TABLE
  workspace.silver.dim_date
ALTER COLUMN
  date_key
  SET NOT NULL;

ALTER TABLE
  workspace.silver.dim_date
ADD
  CONSTRAINT pk_dim_date PRIMARY KEY (date_key);

In [0]:
DESCRIBE TABLE EXTENDED workspace.silver.dim_date;

col_name,data_type,comment
date_key,int,null
full_date,date,null
year,int,null
month,int,null
day,int,null
weekday_name,string,null
is_weekend,boolean,null
# Clustering Information,,
# col_name,data_type,comment
,,


### 1.5 · dim_player
`player_id` is a hex string from outside, so it stays `STRING`. The column list is checked first, then the key and the name column (non-null and not blank).

In [0]:
SELECT
  ordinal_position,
  column_name,
  data_type
FROM
  workspace.information_schema.columns
WHERE
  table_schema = 'bronze'
  AND table_name = 'dim_player'
ORDER BY
  ordinal_position;

ordinal_position,column_name,data_type
0,player_id,STRING
1,player_name,STRING


In [0]:
CREATE OR REPLACE TABLE workspace.silver.dim_player AS
SELECT
  player_id,
  player_name
FROM
  workspace.bronze.dim_player;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  count(*) AS total_rows,
  count(player_id) AS non_null_keys,
  count(DISTINCT player_id) AS distinct_keys,
  count(player_name) AS non_null_names,
  count_if(trim(player_name) = '') AS blank_names
FROM
  workspace.silver.dim_player;

total_rows,non_null_keys,distinct_keys,non_null_names,blank_names
964,964,964,964,0


In [0]:
ALTER TABLE
  workspace.silver.dim_player
ALTER COLUMN
  player_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.dim_player
ADD
  CONSTRAINT pk_dim_player PRIMARY KEY (player_id);

DESCRIBE TABLE EXTENDED workspace.silver.dim_player;

col_name,data_type,comment
player_id,string,null
player_name,string,null
# Clustering Information,,
# col_name,data_type,comment
,,
# Delta Statistics Columns,,
Column Names,"player_id, player_name",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


## 2 · Fact tables

### 2.1 · fact_match
Grain: one row per match; primary key `match_id` (`STRING`). The column list comes from `information_schema`.

In [0]:
SELECT
  ordinal_position,
  column_name,
  data_type
FROM
  workspace.information_schema.columns
WHERE
  table_schema = 'bronze'
  AND table_name = 'fact_match'
ORDER BY
  ordinal_position;

ordinal_position,column_name,data_type
0,match_id,STRING
1,season_id,INT
2,venue_id,INT
3,team1_id,INT
4,team2_id,INT
5,winner_team_id,INT
6,result,STRING
7,eliminator_team_id,INT
8,date_key,INT


In [0]:
CREATE OR REPLACE TABLE workspace.silver.fact_match AS
SELECT
  match_id,
  season_id,
  venue_id,
  team1_id,
  team2_id,
  winner_team_id,
  result,
  eliminator_team_id,
  date_key
FROM
  workspace.bronze.fact_match;

num_affected_rows,num_inserted_rows


Key check, plus the winner and result columns: 1,243 matches, 1,218 with a winner, 25 with a `result`.

In [0]:
SELECT
  count(*) AS total_rows,
  count(match_id) AS non_null_keys,
  count(DISTINCT match_id) AS distinct_keys,
  count(winner_team_id) AS non_null_winners,
  count(result) AS non_null_results
FROM
  workspace.silver.fact_match;

total_rows,non_null_keys,distinct_keys,non_null_winners,non_null_results
1243,1243,1243,1218,25


**Business rule.** `winner_team_id` is NULL exactly when `result` is filled: the 1,218 matches with a winner have no `result`, and the 25 without a winner always have one. The two columns never overlap and never leave a match unexplained.

In [0]:
SELECT
  winner_team_id IS NULL AS winner_is_null,
  result IS NULL AS result_is_null,
  count(*) AS matches
FROM
  workspace.silver.fact_match
GROUP BY
  winner_team_id IS NULL,
  result IS NULL
ORDER BY
  winner_is_null,
  result_is_null;

winner_is_null,result_is_null,matches
false,true,1218
true,false,25


The 25 results are **16 ties** and **9 no-results**.

In [0]:
SELECT
  result,
  count(*) AS matches
FROM
  workspace.silver.fact_match
WHERE
  result IS NOT NULL
GROUP BY
  result;

result,matches
tie,16
no result,9


Add `NOT NULL` and the primary key, then `DESCRIBE`.

In [0]:
ALTER TABLE
  workspace.silver.fact_match
ALTER COLUMN
  match_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.fact_match
ADD
  CONSTRAINT pk_fact_match PRIMARY KEY (match_id);

DESCRIBE TABLE EXTENDED workspace.silver.fact_match;

col_name,data_type,comment
match_id,string,null
season_id,int,null
venue_id,int,null
team1_id,int,null
team2_id,int,null
winner_team_id,int,null
result,string,null
eliminator_team_id,int,null
date_key,int,null
# Clustering Information,,


### 2.2 · fact_delivery
Grain: one row per delivery (295,732 rows). The key is composite, `match_id + innings_number + delivery_seq`. All 16 columns are listed by name in the order returned by `information_schema`.

In [0]:
SELECT
  ordinal_position,
  column_name,
  data_type
FROM
  workspace.information_schema.columns
WHERE
  table_schema = 'bronze'
  AND table_name = 'fact_delivery'
ORDER BY
  ordinal_position;

ordinal_position,column_name,data_type
0,match_id,STRING
1,innings_number,INT
2,delivery_seq,INT
3,is_legal_delivery,BOOLEAN
4,runs_batter,INT
5,runs_extras,INT
6,runs_total,INT
7,is_wicket,BOOLEAN
8,wicket_kind,STRING
9,is_bowler_wicket,BOOLEAN


In [0]:
CREATE OR REPLACE TABLE workspace.silver.fact_delivery AS
SELECT
  match_id,
  innings_number,
  delivery_seq,
  is_legal_delivery,
  runs_batter,
  runs_extras,
  runs_total,
  is_wicket,
  wicket_kind,
  is_bowler_wicket,
  batter_id,
  bowler_id,
  non_striker_id,
  batting_team_id,
  over_number,
  player_out_id
FROM
  workspace.bronze.fact_delivery;

num_affected_rows,num_inserted_rows


Key check: no NULL in any key part, and `count(DISTINCT match_id, innings_number, delivery_seq)` equals the row count (295,732).

In [0]:
SELECT
  count(*) AS total_rows,
  count_if(
    match_id IS NULL
    OR innings_number IS NULL
    OR delivery_seq IS NULL
  ) AS rows_with_null_key,
  count(DISTINCT match_id, innings_number, delivery_seq) AS distinct_keys
FROM
  workspace.silver.fact_delivery;

total_rows,rows_with_null_key,distinct_keys
295732,0,295732


Each part of a composite key needs its own `NOT NULL`, then the constraint is added on all three columns together.

In [0]:
ALTER TABLE
  workspace.silver.fact_delivery
ALTER COLUMN
  match_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.fact_delivery
ALTER COLUMN
  innings_number
  SET NOT NULL;

ALTER TABLE
  workspace.silver.fact_delivery
ALTER COLUMN
  delivery_seq
  SET NOT NULL;

ALTER TABLE
  workspace.silver.fact_delivery
ADD
  CONSTRAINT pk_fact_delivery PRIMARY KEY (match_id, innings_number, delivery_seq);

DESCRIBE TABLE EXTENDED workspace.silver.fact_delivery;

col_name,data_type,comment
match_id,string,null
innings_number,int,null
delivery_seq,int,null
is_legal_delivery,boolean,null
runs_batter,int,null
runs_extras,int,null
runs_total,int,null
is_wicket,boolean,null
wicket_kind,string,null
is_bowler_wicket,boolean,null


## 3 · Alias tables
Raw spelling → generated id (`season_alias`, `team_alias`, `venue_alias`). They let a name that appears in different spellings in the source data resolve to **one** dimension row. The key is the **spelling column**, not the id, because many spellings can point at the same id.

In [0]:
SELECT
  table_name,
  ordinal_position,
  column_name,
  data_type
FROM
  workspace.information_schema.columns
WHERE
  table_schema = 'bronze'
  AND table_name IN ('bridge_wicket_fielder', 'venue_alias', 'team_alias', 'season_alias')
ORDER BY
  table_name,
  ordinal_position;

table_name,ordinal_position,column_name,data_type
bridge_wicket_fielder,0,match_id,STRING
bridge_wicket_fielder,1,innings_number,INT
bridge_wicket_fielder,2,delivery_seq,INT
bridge_wicket_fielder,3,fielder_id,STRING
bridge_wicket_fielder,4,is_substitute,BOOLEAN
season_alias,0,season_label,STRING
season_alias,1,season_id,INT
team_alias,0,team_spelling,STRING
team_alias,1,team_id,INT
venue_alias,0,venue_spelling,STRING


In [0]:
CREATE OR REPLACE TABLE workspace.silver.season_alias AS
SELECT
  season_label,
  season_id
FROM
  workspace.bronze.season_alias;

CREATE OR REPLACE TABLE workspace.silver.team_alias AS
SELECT
  team_spelling,
  team_id
FROM
  workspace.bronze.team_alias;

CREATE OR REPLACE TABLE workspace.silver.venue_alias AS
SELECT
  venue_spelling,
  venue_id
FROM
  workspace.bronze.venue_alias;

num_affected_rows,num_inserted_rows


Key check on all three tables in one result: 19, 19 and 60 rows; spellings unique; ids non-null.

In [0]:
SELECT
  'season_alias' AS tbl,
  count(*) AS total_rows,
  count(season_label) AS non_null_keys,
  count(DISTINCT season_label) AS distinct_keys,
  count(season_id) AS non_null_ids
FROM
  workspace.silver.season_alias
UNION ALL
SELECT
  'team_alias',
  count(*),
  count(team_spelling),
  count(DISTINCT team_spelling),
  count(team_id)
FROM
  workspace.silver.team_alias
UNION ALL
SELECT
  'venue_alias',
  count(*),
  count(venue_spelling),
  count(DISTINCT venue_spelling),
  count(venue_id)
FROM
  workspace.silver.venue_alias;

tbl,total_rows,non_null_keys,distinct_keys,non_null_ids
season_alias,19,19,19,19
team_alias,19,19,19,19
venue_alias,60,60,60,60


Orphan check: every alias id must exist in its dimension. Expected **0** for each.

In [0]:
SELECT
  'season_alias' AS tbl,
  count(*) AS orphans
FROM
  workspace.silver.season_alias a
    LEFT JOIN workspace.silver.dim_season d
      ON a.season_id = d.season_id
WHERE
  a.season_id IS NOT NULL
  AND d.season_id IS NULL
UNION ALL
SELECT
  'team_alias',
  count(*)
FROM
  workspace.silver.team_alias a
    LEFT JOIN workspace.silver.dim_team d
      ON a.team_id = d.team_id
WHERE
  a.team_id IS NOT NULL
  AND d.team_id IS NULL
UNION ALL
SELECT
  'venue_alias',
  count(*)
FROM
  workspace.silver.venue_alias a
    LEFT JOIN workspace.silver.dim_venue d
      ON a.venue_id = d.venue_id
WHERE
  a.venue_id IS NOT NULL
  AND d.venue_id IS NULL;

tbl,orphans
season_alias,0
team_alias,0
venue_alias,0


`NOT NULL` on both columns, primary key on the spelling column.

In [0]:
ALTER TABLE
  workspace.silver.season_alias
ALTER COLUMN
  season_label
  SET NOT NULL;

ALTER TABLE
  workspace.silver.season_alias
ALTER COLUMN
  season_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.season_alias
ADD
  CONSTRAINT pk_season_alias PRIMARY KEY (season_label);

ALTER TABLE
  workspace.silver.team_alias
ALTER COLUMN
  team_spelling
  SET NOT NULL;

ALTER TABLE
  workspace.silver.team_alias
ALTER COLUMN
  team_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.team_alias
ADD
  CONSTRAINT pk_team_alias PRIMARY KEY (team_spelling);

ALTER TABLE
  workspace.silver.venue_alias
ALTER COLUMN
  venue_spelling
  SET NOT NULL;

ALTER TABLE
  workspace.silver.venue_alias
ALTER COLUMN
  venue_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.venue_alias
ADD
  CONSTRAINT pk_venue_alias PRIMARY KEY (venue_spelling);

## 4 · bridge_wicket_fielder
A bridge table: one wicket can involve more than one fielder, so the key is the delivery key **plus** `fielder_id`. The key check shows 11,266 rows but only **10,721 distinct deliveries**: the 545 extra rows are deliveries with more than one fielder. That is why `fielder_id` must be part of the key.

In [0]:
CREATE OR REPLACE TABLE workspace.silver.bridge_wicket_fielder AS
SELECT
  match_id,
  innings_number,
  delivery_seq,
  fielder_id,
  is_substitute
FROM
  workspace.bronze.bridge_wicket_fielder;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  count(*) AS total_rows,
  count_if(
    match_id IS NULL
    OR innings_number IS NULL
    OR delivery_seq IS NULL
    OR fielder_id IS NULL
  ) AS rows_with_null_key,
  count(DISTINCT match_id, innings_number, delivery_seq) AS distinct_deliveries,
  count(DISTINCT match_id, innings_number, delivery_seq, fielder_id) AS distinct_with_fielder
FROM
  workspace.silver.bridge_wicket_fielder;

total_rows,rows_with_null_key,distinct_deliveries,distinct_with_fielder
11266,0,10721,11266


Four-column primary key, each column `NOT NULL`.

In [0]:
ALTER TABLE
  workspace.silver.bridge_wicket_fielder
ALTER COLUMN
  match_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.bridge_wicket_fielder
ALTER COLUMN
  innings_number
  SET NOT NULL;

ALTER TABLE
  workspace.silver.bridge_wicket_fielder
ALTER COLUMN
  delivery_seq
  SET NOT NULL;

ALTER TABLE
  workspace.silver.bridge_wicket_fielder
ALTER COLUMN
  fielder_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.bridge_wicket_fielder
ADD
  CONSTRAINT pk_bridge_wicket_fielder
    PRIMARY KEY (match_id, innings_number, delivery_seq, fielder_id);

# Part C · Final verification

**Every table has a primary key.** `information_schema.table_constraints` should list 11 rows, one per silver table.

In [0]:
SELECT
  table_name,
  constraint_name,
  constraint_type
FROM
  workspace.information_schema.table_constraints
WHERE
  table_schema = 'silver'
ORDER BY
  table_name;

table_name,constraint_name,constraint_type
bridge_wicket_fielder,pk_bridge_wicket_fielder,PRIMARY KEY
dim_date,pk_dim_date,PRIMARY KEY
dim_player,pk_dim_player,PRIMARY KEY
dim_season,pk_dim_season,PRIMARY KEY
dim_team,pk_dim_team,PRIMARY KEY
dim_venue,pk_dim_venue,PRIMARY KEY
fact_delivery,pk_fact_delivery,PRIMARY KEY
fact_match,pk_fact_match,PRIMARY KEY
season_alias,pk_season_alias,PRIMARY KEY
team_alias,pk_team_alias,PRIMARY KEY


**Row counts, bronze vs silver.** Silver is a validated copy, so every pair must match.

In [0]:
SELECT
  'dim_season' AS tbl,
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.dim_season
  ) AS bronze_rows,
  (
    SELECT
      count(*)
    FROM
      workspace.silver.dim_season
  ) AS silver_rows
UNION ALL
SELECT
  'dim_team',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.dim_team
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.dim_team
  )
UNION ALL
SELECT
  'dim_venue',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.dim_venue
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.dim_venue
  )
UNION ALL
SELECT
  'dim_date',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.dim_date
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.dim_date
  )
UNION ALL
SELECT
  'dim_player',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.dim_player
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.dim_player
  )
UNION ALL
SELECT
  'fact_match',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.fact_match
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.fact_match
  )
UNION ALL
SELECT
  'fact_delivery',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.fact_delivery
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.fact_delivery
  )
UNION ALL
SELECT
  'bridge_wicket_fielder',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.bridge_wicket_fielder
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.bridge_wicket_fielder
  )
UNION ALL
SELECT
  'season_alias',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.season_alias
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.season_alias
  )
UNION ALL
SELECT
  'team_alias',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.team_alias
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.team_alias
  )
UNION ALL
SELECT
  'venue_alias',
  (
    SELECT
      count(*)
    FROM
      workspace.bronze.venue_alias
  ),
  (
    SELECT
      count(*)
    FROM
      workspace.silver.venue_alias
  );

tbl,bronze_rows,silver_rows
dim_season,19,19
dim_team,15,15
dim_venue,36,36
dim_date,6618,6618
dim_player,964,964
fact_match,1243,1243
fact_delivery,295732,295732
bridge_wicket_fielder,11266,11266
season_alias,19,19
team_alias,19,19


## Silver complete
All 11 tables are built in `workspace.silver` with explicit schemas, `NOT NULL` keys and primary keys, and every check above passed:
- the two ways of totalling runs agree (401,738), and wickets reconcile (14,705);
- all foreign-key orphan checks return 0;
- every key is non-null and unique;
- `dim_date` is internally consistent;
- row counts match bronze on all 11 tables.

Gold (analysis queries) reads from these tables.

# Appendix · What Unity Catalog constraints enforce

A demonstration on `dim_season`, then a reset. It shows why the key checks above use `count(DISTINCT ...)` instead of trusting `PRIMARY KEY`.

- `NOT NULL` **is enforced**: an insert with a NULL key is rejected (the last cell of the notebook).
- `PRIMARY KEY` **is informational only**: inserting a duplicate key succeeds.

**Step 1: insert a duplicate key.** The row copies an existing row. The insert succeeds even though `season_id` is a primary key.

In [0]:
INSERT INTO workspace.silver.dim_season
  SELECT
    season_id,
    season_year
  FROM
    workspace.silver.dim_season
  LIMIT 1;

num_affected_rows,num_inserted_rows
1,1


The key check now shows **20 rows but 19 distinct keys**. The constraint did not stop the duplicate; only this kind of check can find it.

In [0]:
SELECT
  count(*) AS total_rows,
  count(DISTINCT season_id) AS distinct_keys
FROM
  workspace.silver.dim_season;

total_rows,distinct_keys
19,19


**Step 2: reset.** `CREATE OR REPLACE TABLE` rebuilds `dim_season` from bronze (this also drops the constraints), the key check confirms 19 / 19, and the constraints are added again.

In [0]:
CREATE OR REPLACE TABLE workspace.silver.dim_season AS
SELECT
  season_id,
  season_year
FROM
  workspace.bronze.dim_season;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  count(*) AS total_rows,
  count(DISTINCT season_id) AS distinct_keys
FROM
  workspace.silver.dim_season;

total_rows,distinct_keys
19,19


In [0]:
ALTER TABLE
  workspace.silver.dim_season
ALTER COLUMN
  season_id
  SET NOT NULL;

ALTER TABLE
  workspace.silver.dim_season
ADD
  CONSTRAINT pk_dim_season PRIMARY KEY (season_id);

**Step 3: NOT NULL is enforced. This cell is expected to fail.**
The insert below is rejected with `DELTA_NOT_NULL_CONSTRAINT_VIOLATED`, and nothing is written. It is the last cell, so *Run all* completes everything above it.

In [0]:
INSERT INTO workspace.silver.dim_season
  VALUES (NULL, 2099);

DeltaInvariantViolationException: [DELTA_NOT_NULL_CONSTRAINT_VIOLATED] NOT NULL constraint violated for column: season_id.